In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV
)

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.svm import SVR

from sklearn.metrics import (
    r2_score,
    mean_squared_error
)

import plotly.express as px

import joblib

In [2]:
df = pd.read_csv(
    "../data/processed/featured_dataset.csv"
)

df.head()

,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,HOUR,DAY,MONTH,DAY_OF_WEEK,WEEK_OF_YEAR,IS_PEAK_HOUR,TEMP_DIFFERENCE,IRRADIATION_EFFICIENCY,PREVIOUS_AC_POWER,ROLLING_MEAN_AC_POWER
0,2020-05-15 00:00:00,4136001,LYwnQax7tkwH5Cb,0.0,0.0,1872.500000,1.794959e+09,27.004764,25.060789,0.0,0,15,5,4,20,0,-1.943975,0.0,0.0,0.0
1,2020-05-15 00:00:00,4136001,LlT2YUhhzqhg5Sw,0.0,0.0,1094.357143,2.825928e+08,27.004764,25.060789,0.0,0,15,5,4,20,0,-1.943975,0.0,0.0,0.0
2,2020-05-15 00:00:00,4136001,Mx2yZCDsyf6DPfv,0.0,0.0,5692.200000,2.453646e+06,27.004764,25.060789,0.0,0,15,5,4,20,0,-1.943975,0.0,0.0,0.0
3,2020-05-15 00:00:00,4136001,NgDl19wMapZy17u,0.0,0.0,1866.200000,1.115126e+08,27.004764,25.060789,0.0,0,15,5,4,20,0,-1.943975,0.0,0.0,0.0
4,2020-05-15 00:00:00,4136001,PeE6FRyGXUgsRhN,0.0,0.0,651.200000,1.348351e+09,27.004764,25.060789,0.0,0,15,5,4,20,0,-1.943975,0.0,0.0,0.0


In [3]:
X = df.drop(
    columns=[
        'AC_POWER',
        'DATE_TIME',
        'SOURCE_KEY',
        'DC_POWER'
    ]
)

y = df['AC_POWER']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [6]:
svr_model = SVR()

In [7]:
param_grid_svr = {

    'kernel': [
        'rbf',
        'linear'
    ],

    'C': [
        0.1,
        1,
        10,
        100
    ],

    'gamma': [
        'scale',
        'auto'
    ],

    'epsilon': [
        0.01,
        0.1,
        0.5
    ]
}

In [8]:
svr_random = RandomizedSearchCV(

    estimator=svr_model,

    param_distributions=param_grid_svr,

    n_iter=5,

    cv=3,

    verbose=2,

    random_state=42,

    n_jobs=-1
)

In [9]:
svr_random.fit(

    X_train_scaled,

    y_train
)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


KeyboardInterrupt: 

In [ ]:
print(
    "Best SVR Parameters:"
)

print(
    svr_random.best_params_
)

In [ ]:
best_svr_model = svr_random.best_estimator_

y_pred = best_svr_model.predict(
    X_test_scaled
)

In [ ]:
r2 = r2_score(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("R² Score:", r2)

print("RMSE:", rmse)

In [ ]:
joblib.dump(

    best_svr_model,

    "../models/svr_model.pkl"
)

In [ ]:
svr_results = pd.DataFrame(
    svr_random.cv_results_
)

display(

    svr_results[
        [
            'param_kernel',
            'param_C',
            'param_gamma',
            'param_epsilon',
            'mean_test_score',
            'rank_test_score'
        ]
    ].sort_values(
        by='rank_test_score'
    ).head(10)
)

fig1 = px.line(

    svr_results,

    x='param_C',

    y='mean_test_score',

    color='param_kernel',

    title='SVR: C vs Mean CV Score',

    markers=True
)

fig1.show()

fig2 = px.box(

    svr_results,

    x='param_epsilon',

    y='mean_test_score',

    title='SVR: epsilon vs CV Score'
)

fig2.show()

fig3 = px.box(

    svr_results,

    x='param_kernel',

    y='mean_test_score',

    title='SVR: kernel comparison'
)

fig3.show()